# Fusion + GNN Switching inspection

Notebook pour voir et comparer le pipeline actuel:

- verifier le GNN GraphSAGE switching exporte;
- comparer `identity` vs `random_frozen_tcn`;
- inspecter les sorties `fusion_outputs.jsonl`;
- verifier les shapes `[64]`, `[1,64]`, `[1,512]`, `[1,12]`;
- regarder les normes L2, cold starts, metadata et notification ONNX.

Important: ce notebook ne modifie pas les poids GNN, ne re-entraine pas le GNN et n'utilise pas le decoder.

## 1. Setup

Lancer ce notebook depuis la racine du projet ou depuis `notebooks/`. Le code retrouve automatiquement la racine contenant `fusion_model.py`.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import torch

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "fusion_model.py").is_file():
            return candidate
    raise RuntimeError("Could not find project root containing fusion_model.py")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "fusion_smoke"
SWITCHING_EXPORT_DIR = ROOT / "pre_embedders" / "switching" / "exports" / "switching_encoder"

print("ROOT:", ROOT)
print("DATA_DIR exists:", DATA_DIR.exists())
print("OUTPUT_DIR exists:", OUTPUT_DIR.exists())
print("SWITCHING_EXPORT_DIR exists:", SWITCHING_EXPORT_DIR.exists())

## 2. Inspecter les sessions et les graphes 120s

In [ ]:
sessions = sorted(p for p in DATA_DIR.glob("session_*") if p.is_dir())
graphs = sorted(DATA_DIR.glob("session_*/data_graph/data_graph_120s/graph_*.json"))

print("sessions:", len(sessions))
for session in sessions[:10]:
    n_graphs = len(list(session.glob("data_graph/data_graph_120s/graph_*.json")))
    print(f"- {session.name}: {n_graphs} graphes 120s")

print("total graphes 120s:", len(graphs))
print("premier graphe:", graphs[0] if graphs else None)

## 3. Charger le GNN switching une seule fois

Ce chargement appelle le package exporte `output.py`. Le decoder n'est pas appele.

In [ ]:
from pre_embedders.switching import load_model as load_switching_model

switching_session = load_switching_model(export_dir=SWITCHING_EXPORT_DIR, device="cpu")
print(type(switching_session).__name__)
print("export_dir:", switching_session.export_dir)
print("device:", switching_session.device)

## 4. Tester un vrai graphe avec le GNN

Sortie attendue: embedding `[64]`, `float32`, L2 norm proche de `1.0`, `decoder_used_at_inference=False`.

In [ ]:
graph_path = graphs[0] if graphs else None
if graph_path is None:
    raise FileNotFoundError("No graph_*.json found under data/session_*/data_graph/data_graph_120s")

switching_payload = switching_session.get_fusion_input(graph_path)
embedding = switching_payload["embedding"]
metadata = switching_payload["metadata"]

print("graph:", graph_path)
print("embedding shape:", embedding.shape)
print("embedding dtype:", embedding.dtype)
print("embedding L2 norm:", float(np.linalg.norm(embedding)))
print("cold_start:", metadata.get("cold_start"))
print("decoder_used_at_inference:", metadata.get("decoder_used_at_inference"))
print(json.dumps(metadata, indent=2, sort_keys=True))

## 5. Comparer identity-only vs random-frozen TCN

Le main path doit rester `identity` avec une sortie `[1,64]`. Le `random_frozen_tcn` doit rester seulement une ablation avec une sortie `[1,32]`.

In [ ]:
from TCN_encoders.switching.encoder import SwitchingBufferedEncoder

identity_encoder = SwitchingBufferedEncoder(mode="identity")
random_frozen_encoder = SwitchingBufferedEncoder(mode="random_frozen_tcn")

h_identity, freshness_identity = identity_encoder.step(switching_payload)
h_random, freshness_random = random_frozen_encoder.step(switching_payload)

comparison = [
    {
        "mode": "identity",
        "shape": tuple(h_identity.shape),
        "freshness": freshness_identity,
        "debug_mode": identity_encoder.debug_state().get("mode"),
    },
    {
        "mode": "random_frozen_tcn",
        "shape": tuple(h_random.shape),
        "freshness": freshness_random,
        "debug_mode": random_frozen_encoder.debug_state().get("mode"),
    },
]

if pd:
    display(pd.DataFrame(comparison))
else:
    print(comparison)

## 6. Inspecter la fusion et les predictive models actifs

`512` vient de Tucker: `rank^3 = 8^3`.

In [ ]:
from fusion_model import DEFAULT_D_DIMS, InferrerFusion

fusion = InferrerFusion()
fusion.eval()

print("DEFAULT_D_DIMS:", DEFAULT_D_DIMS)
print("TFN class:", type(fusion.tfn).__name__)
print("rank:", fusion.tfn.rank)
print("flat sizes:", [fusion.tfn.flat_size(i) for i in range(4)])
print("predictive models:", [type(m).__name__ for m in fusion.models])

In [ ]:
embeddings = [
    torch.zeros(1, 64),  # mouse dummy/current
    torch.zeros(1, 64),  # keyboard dummy/current
    torch.zeros(1, 32),  # notif dummy/current
    h_identity,          # real switching identity tensor [1,64]
]

with torch.no_grad():
    tucker_tensor = fusion.tfn(embeddings)
    slices = [fusion.tfn.get_slice(tucker_tensor, i) for i in range(4)]
    output = fusion(embeddings)

print("tucker tensor shape:", tuple(tucker_tensor.shape))
print("slice shapes:", [tuple(s.shape) for s in slices])
print("global shape:", tuple(output["global"].shape))
print("per_model shapes:", [tuple(t.shape) for t in output["per_model"]])

## 7. Lire et comparer les fichiers `fusion_outputs*.jsonl`

Cette cellule resume les outputs deja generes par les smoke tests.

In [ ]:
jsonl_files = sorted(OUTPUT_DIR.glob("**/fusion_outputs*.jsonl"), key=lambda p: p.stat().st_mtime, reverse=True)
print("JSONL files:", len(jsonl_files))
for path in jsonl_files[:10]:
    print("-", path.relative_to(ROOT))


def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


summary_rows = []
for path in jsonl_files:
    rows = load_jsonl(path)
    if not rows:
        continue
    switching_shapes = sorted({str(r.get("switching", {}).get("tensor_shape")) for r in rows})
    embedding_shapes = sorted({str(r.get("switching", {}).get("embedding_shape")) for r in rows})
    norms = [float(r.get("switching", {}).get("embedding_l2_norm", 0.0)) for r in rows]
    cold_starts = sum(1 for r in rows if r.get("switching", {}).get("metadata", {}).get("cold_start") is True)
    notif_sources = sorted({str(r.get("notif", {}).get("metadata", {}).get("embedding_source")) for r in rows})
    summary_rows.append({
        "file": str(path.relative_to(ROOT)),
        "windows": len(rows),
        "switching_embedding_shapes": embedding_shapes,
        "switching_tensor_shapes": switching_shapes,
        "cold_starts": cold_starts,
        "l2_min": min(norms),
        "l2_max": max(norms),
        "notif_sources": notif_sources,
    })

if pd:
    display(pd.DataFrame(summary_rows))
else:
    print(json.dumps(summary_rows, indent=2))

## 8. Visualiser une session JSONL

Par defaut, on prend le fichier le plus recent.

In [ ]:
selected_jsonl = jsonl_files[0] if jsonl_files else None
if selected_jsonl is None:
    raise FileNotFoundError("No fusion_outputs*.jsonl found under outputs/fusion_smoke")

rows = load_jsonl(selected_jsonl)
records = []
for r in rows:
    sw = r.get("switching", {})
    notif = r.get("notif", {})
    fusion_out = r.get("fusion", {})
    records.append({
        "window_index": r.get("window_index"),
        "window_id": r.get("window_id"),
        "graph_id": r.get("graph_id"),
        "switching_l2": sw.get("embedding_l2_norm"),
        "switching_embedding_shape": sw.get("embedding_shape"),
        "switching_tensor_shape": sw.get("tensor_shape"),
        "cold_start": sw.get("metadata", {}).get("cold_start"),
        "notif_source": notif.get("metadata", {}).get("embedding_source"),
        "notif_tensor_shape": notif.get("tensor_shape"),
        "global_shape": fusion_out.get("global_shape"),
        "per_model_shapes": fusion_out.get("per_model_shapes"),
    })

print("selected:", selected_jsonl.relative_to(ROOT))
if pd:
    df = pd.DataFrame(records)
    display(df)
else:
    print(json.dumps(records[:5], indent=2))

In [ ]:
if pd and plt:
    ax = df.plot(x="window_index", y="switching_l2", marker="o", figsize=(8, 3), title="Switching embedding L2 norm par fenetre")
    ax.axhline(1.0, color="black", linewidth=1, linestyle="--")
    ax.set_ylim(0.95, 1.02)
    ax.set_ylabel("L2 norm")
    plt.show()
else:
    print("Install pandas/matplotlib to display plots.")

## 9. Comparer plusieurs embeddings GNN

Cette cellule calcule les embeddings switching sur les premiers graphes et affiche une matrice de similarite cosinus. Comme les embeddings sont L2-normalises, le produit scalaire est une similarite cosinus.

In [ ]:
N = min(8, len(graphs))
embedding_rows = []
embedding_matrix = []

for graph in graphs[:N]:
    payload = switching_session.get_fusion_input(graph)
    emb = payload["embedding"]
    meta = payload["metadata"]
    embedding_matrix.append(emb)
    embedding_rows.append({
        "graph": graph.name,
        "session_id": meta.get("session_id"),
        "window_id": meta.get("window_id"),
        "cold_start": meta.get("cold_start"),
        "l2_norm": float(np.linalg.norm(emb)),
    })

if pd:
    display(pd.DataFrame(embedding_rows))
else:
    print(embedding_rows)

if embedding_matrix:
    E = np.vstack(embedding_matrix)
    cosine = E @ E.T
    if pd:
        display(pd.DataFrame(cosine, index=[r["graph"] for r in embedding_rows], columns=[r["graph"] for r in embedding_rows]))
    else:
        print(cosine)

In [ ]:
if plt and embedding_matrix:
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title("Cosine similarity entre embeddings GNN")
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels([r["graph"] for r in embedding_rows], rotation=45, ha="right")
    ax.set_yticklabels([r["graph"] for r in embedding_rows])
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib to display heatmap.")

## 10. Commandes de tests depuis le notebook

Les cellules suivantes peuvent etre executees pour relancer les tests. Elles sont separees pour garder le notebook leger.

In [ ]:
# Relancer les tests unitaires fusion
# !python test_fusion.py

In [ ]:
# Relancer les tests integration switching/GNN
# !python -m unittest tests.test_switching_encoder_integration

In [ ]:
# Generer un nouveau JSONL smoke sur 2 fenetres
# !python run_fusion_session_smoke.py --session-dir data/session_20260516_105702_81c740 --limit 2